# Additive attention for an encoder-decoder

Before dot-product attention, encoder-decoder models scored each source position with a small feed-forward network: additive (Bahdanau-style) attention. Implement it for all decoder steps at once.

Write `additive_attention(query, keys, values, W_q, W_k, v, src_mask)`:

- `query`: float64 `(B, T, Dq)`, one query per decoder step.
- `keys`: float64 `(B, S, Dk)`, `values`: float64 `(B, S, Dv)`: encoder states.
- `W_q`: `(A, Dq)`, `W_k`: `(A, Dk)`, `v`: `(A,)`: the scoring network (no biases).
- `src_mask`: `(B, S)` of 0/1 or bool; 1 marks a real source token, 0 padding.

For each batch $b$, step $t$ and source position $s$:

$$e_{bts} = v^\top \tanh\left(W_q\, q_{bt} + W_k\, k_{bs}\right)$$

Padded positions get no weight. The attention weights are $\alpha_{bt\cdot} = \text{softmax}_s(e_{bt\cdot})$ over the real positions, and the context is $c_{bt} = \sum_s \alpha_{bts}\, \text{values}_{bs}$.

Return a tuple `(context, weights)` of shapes `(B, T, Dv)` and `(B, T, S)`. If a sequence has no real source token, its weights and context are all zeros (not NaN).

Use broadcasting, no Python loops over `B`, `T` or `S`. Project the keys once, not once per query.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def additive_attention(query, keys, values, W_q, W_k, v, src_mask):
    """Return (context (B, T, Dv), weights (B, T, S))."""
    q = query @ W_q.T                                   # (B, T, A)
    k = keys @ W_k.T                                    # (B, S, A)
    e = torch.tanh(q + k) @ v                           # TODO: shapes don't line up, and padding is ignored
    w = torch.softmax(e, dim=-1)
    return w @ values, w


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[1.0]]], "dtype": "float64"}, {"__nd__": [[[1.0], [-1.0]]], "dtype": "float64"}, {"__nd__": [[[10.0], [20.0]]], "dtype": "float64"}, {"__nd__": [[1.0]], "dtype": "float64"}, {"__nd__": [[1.0]], "dtype": "float64"}, {"__nd__": [1.0], "dtype": "float64"}, {"__nd__": [[1, 1]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[12.760725313359536]]], "dtype": "float64"}, {"__nd__": [[[0.7239274686640463, 0.2760725313359536]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.0], [2.0]]], "dtype": "float64"}, {"__nd__": [[[1.0], [0.0], [5.0]]], "dtype": "float64"}, {"__nd__": [[[1.0, 0.0], [0.0, 1.0], [9.0, 9.0]]], "dtype": "float64"}, {"__nd__": [[1.0], [-1.0]], "dtype": "float64"}, {"__nd__": [[1.0], [1.0]], "dtype": "float64"}, {"__nd__": [1.0, 1.0], "dtype": "float64"}, {"__nd__": [[1, 1, 0]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[0.8210074960059999, 0.1789925039940001], [0.5581014925883542, 0.441898507411646]]], "dtype": "float64"}, {"__nd__": [[[0.8210074960059999, 0.1789925039940001, 0.0], [0.5581014925883542, 0.441898507411646, 0.0]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.5532605888887387, 0.21760060921947388, -0.057989987866526184], [-2.3189360947215656, 0.4314941677088505, -2.126279784450882], [0.9099212170226342, 0.6059655730064136, 0.8300566485784159]], [[0.8276983437153878, 0.2985144698332214, -0.5350014137339273], [-0.3070622870057959, 1.5080722715721342, -0.5822293778059019], [-0.22812380385128453, -0.7245145044162855, -0.5172493374907822]]], "dtype": "float64"}, {"__nd__": [[[-0.30655788195456973, 0.2565516458675206], [-0.2938560550257995, -0.3547084649887684], [-0.6168151525371501, 0.08958446690608873], [-1.3441660008066356, 0.052184899307776686]], [[1.3100257323485311, -0.7669128788338543], [-0.04539173803348671, 2.817209016484243], [-0.9926744097609564, -1.621141524211868], [-0.42680913157992273, 1.8069379803781687]]], "dtype": "float64"}, {"__nd__": [[[2.035115511784091, -1.2159872631305197], [0.5701405884703373, 0.004521431343134918], [-0.5883707554353481, -1.9456978173547614], [1.0245839797071592, -0.8605383067300788]], [[0.29900768417420936, 0.08189396317057414], [1.8701812688972848, 1.1759947894135871], [-1.1553360632608776, -0.2623951558776075], [1.0368475527289776, -1.333014964281339]]], "dtype": "float64"}, {"__nd__": [[-0.3922883318669356, 0.22757580070206257, -1.6158261586823717], [-1.2354581464367866, -1.7040770041170938, 0.28697587878613073], [-0.5615144689797945, 0.9286482876454955, 0.11379425356523251], [0.5720172533764897, -0.08367584391494008, 0.7519237952456632], [-0.10331715721652544, -0.9822015508199493, 1.1827683822490698]], "dtype": "float64"}, {"__nd__": [[0.9800081476549486, 2.2666932138811453], [-0.14220093149203353, -0.5799002829800761], [-1.4470425027291827, -3.859277514716476], [-1.308398340087321, 0.48681855045100436], [-0.8855311413697625, -2.5948239990829336]], "dtype": "float64"}, {"__nd__": [0.6082597078488055, -0.5916575081322901, -0.36100409352785295, -0.43143950165476136, -0.13469839560367824], "dtype": "float64"}, {"__nd__": [[1, 1, 1, 1], [1, 1, 0, 0]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[1.1592082377874484, -1.1599691030391281], [0.7931881314593869, -1.0215058178174625], [0.9584454693808163, -1.0803239724825169]], [[1.272306497518412, 0.7596593267277308], [1.1725223916285672, 0.6901737690773698], [1.4856369713015676, 0.9082139166841947]]], "dtype": "float64"}, {"__nd__": [[[0.5347598558947754, 0.13976797464761972, 0.21218872330902233, 0.11328344614858266], [0.27942807353461385, 0.24257413793419488, 0.25018092872717496, 0.22781685980401628], [0.3893980347265282, 0.19461853929148096, 0.23013317476656572, 0.18585025121542512]], [[0.38052750962221016, 0.6194724903777897, 0.0, 0.0], [0.44403679138462754, 0.5559632086153725, 0.0, 0.0], [0.24474972169513304, 0.755250278304867, 0.0, 0.0]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.47444407695595, -1.7345166510884678]]], "dtype": "float64"}, {"__nd__": [[[-0.22711744271967574, -0.5085891107173511], [1.1367247203334006, -1.0352973533071197], [1.6374114454309625, 0.6080331975223119], [1.6246433863782133, -0.7044853751017233], [0.8059521152228815, 0.3631040001715692]]], "dtype": "float64"}, {"__nd__": [[[-1.497670353579111, -0.26091789722820546, -0.4526963321434531], [-0.9661537662976033, -0.7681495562031, -1.4398268099452578], [-0.4671738791781496, 0.3341363029795037, -0.7788386330876441], [-1.5631519863854781, 1.0453300853245673, -0.37081461226831575], [0.027424727629300243, -0.23394341341574568, -0.5401185262342673]]], "dtype": "float64"}, {"__nd__": [[0.6017587397889111, -0.5299062148166881], [0.028727294300772213, -1.4449418453788945], [-1.6387657921658814, -0.38955695537009144], [1.9106715035731066, 0.1812625943452766]], "dtype": "float64"}, {"__nd__": [[1.9336300601288359, 0.4521845499496128], [0.4534593078626402, -0.5833236968021522], [-0.022417566518173702, 1.7912857198271908], [1.1851895804822443, -0.6261892903522636]], "dtype": "float64"}, {"__nd__": [-0.46981695997984013, 1.337241742949975, -0.5147852634626356, 0.9197404973376132], "dtype": "float64"}, {"__nd__": [[1, 0, 1, 0, 1]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[-0.8335710904504386, -0.10202002590880628, -0.5588955801564659]]], "dtype": "float64"}, {"__nd__": [[[0.481819809174047, 0.0, 0.2551054437740159, 0.0, 0.2630747470519371]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.9095622374384135, 0.6280165940626083], [1.5262020846668078, 0.835006130784672]], [[0.24344399384005985, 0.14039998297355255], [-0.08455620551134427, -0.023061339558410495]]], "dtype": "float64"}, {"__nd__": [[[1.1788224355661225, 1.15390491426208, 0.40781216959604366], [-1.2592136077645948, -0.520658641532731, 0.29740088482682187], [0.05149551908664256, -1.3529665793595937, 0.6640093321096796]], [[2.5988309831504903, -1.2265409197470718, 0.06377375209811639], [1.61978856621042, -0.41734810360675173, -0.40774174906919597], [0.16957523098743163, 0.5669816138670667, -0.4111957110197731]]], "dtype": "float64"}, {"__nd__": [[[0.9798143542823354, -1.9049970217947472], [0.655421857244873, 0.36877126586848974], [-0.15890898812285262, 0.38736743152539377]], [[0.04787779318377918, 1.329818670087016], [-0.4617116581712759, -0.04141893043933976], [-1.387654881387399, -1.1247761756336008]]], "dtype": "float64"}, {"__nd__": [[-0.24098292687567094, -1.5560283497926257], [-0.8472880919393981, -1.3260211766736796], [-0.493601948194795, 0.11743839429605914]], "dtype": "float64"}, {"__nd__": [[0.9435369892862817, -1.9406078816289432, -0.20033897046056426], [0.8657073055485704, 1.3801302841729033, 1.4060709502619853], [-1.4343265844965274, -0.5131397260884635, 0.5641874893843708]], "dtype": "float64"}, {"__nd__": [0.9387234898768438, 0.2820188282897474, 0.26558640328963834], "dtype": "float64"}, {"__nd__": [[0, 0, 0], [1, 1, 1]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[0.0, 0.0], [0.0, 0.0]], [[-0.29593351597117995, 0.5290294356197794], [-0.3173167100444776, 0.49131391779331796]]], "dtype": "float64"}, {"__nd__": [[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.48578006201221663, 0.42590971439122477, 0.0883102235965587], [0.4693970489931205, 0.42821564025147735, 0.10238731075540214]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.24109857143139354, -0.767360786337068, 0.16129042571022478, 0.812236820154686], [-0.33907038376846577, 1.1728353212200668, -2.8421345556851825, -0.32651169373316946], [1.700220235506148, 0.8735536573431392, 1.4441728528739437, 0.14612519800614662], [-1.1817241461859231, -0.5104510153909563, 1.52219718663443, -1.8365815440219238]], [[-0.9536317245125848, -1.7139344118290056, -0.5336349645185658, -0.4212379900574514], [0.5799363404994193, -0.823915421428668, 1.7135000303244519, 0.6477273898405688], [-0.47090516987550435, -0.9910645338301242, -1.4978566295734526, 0.12336137414890104], [1.983714638212815, -0.16046464512974998, -1.0261433361766163, -0.05588164653826363]], [[-0.5139143393319071, -0.901469713862993, -1.893521073060943, -0.053416097268674], [1.4888947585205548, 1.2314825317133544, 0.9875729460028575, 0.21489765417894124], [0.47746618437907484, 1.0171842958596853, -0.047759930102910526, 0.2893196184637676], [-0.3914239053578755, 1.140783208584205, -0.9182083587191213, 0.6901000340614477]]], "dtype": "float64"}, {"__nd__": [[[0.8650198290418158, 0.8460553055003698, 0.7213915093385939, -0.4541162569906305], [-0.5537050377425773, -0.40141320828001226, -0.3641275684342096, -1.1150614694493721], [-0.4344430888960559, -0.6253801681907678, -0.9368719848731346, 0.3756749135768645], [-0.744254517322277, 0.2624993733399629, 1.48677631252472, 1.7028700586993812], [0.03404252264278468, 1.0381714995744853, 0.2666570820287929, 2.4109596090001237], [1.6655972856414405, 1.1367963412352344, -0.13834614130453993, -0.23239173752978795]], [[0.5663440245740382, -0.013940134104685085, -0.2286840592004568, 0.7156772374312473], [-1.2335605384426214, -1.9892985863636607, -1.0032833436493345, -1.1447900189737565], [-0.06800987354659407, -0.8592685641415428, -0.18273095943168408, -2.01759274191572], [1.489928322201855, 0.4106974578461337, -0.8554072499021402, 0.32953840860265093], [1.248591231184819, 0.8575121266794775, -0.07836388230610881, 0.6296101058342439], [0.901539067592392, -1.2027824036812047, -1.4354318442447123, -0.5858865853099267]], [[1.0009539605743045, 1.4963227011742488, 0.14631363979616674, 1.5255658786331128], [1.016129281776467, -0.059700986218757565, -0.7373328349120187, -0.6347468957174698], [0.09383003523679159, 1.4256720488464023, 0.8571510352635652, -1.3003236775472613], [-1.2231372636159012, 0.8309051921212542, 0.21027028626919175, -0.9396619893465503], [0.6950614168575278, 1.3884080538511916, 0.8171475302908839, 0.12215804726956613], [1.3605958857826965, -0.3735580691036685, -0.5804498795997574, -0.9129789298634137]]], "dtype": "float64"}, {"__nd__": [[[1.9217032637244162, 2.1532999663605445, -0.024281463734012982], [0.6140634482474299, 0.47401865862733406, 0.9607584696141868], [1.2197283900354003, 0.705481979187263, -0.07205161473307659], [-0.5003237058261072, -1.0049263685016512, -2.6094582134903725], [-0.07576550886977991, -0.817176625060308, 0.05142107609694558], [-0.0847632381316415, -1.012488428139888, -0.48338356105366587]], [[0.43655552728884944, 0.18971776597643028, -0.8664518380813803], [-0.7378674254288081, 0.5928539404329747, 0.2585746116795385], [0.32925735550295243, -0.144748560010324, 0.7629107587422029], [0.11412035721971718, 2.557112209761941, -0.454526610983739], [-0.6871369880154431, 1.6766412459746578, -0.05408543460530651], [-0.7640969066116208, 0.45277685963891395, -2.3062521239791374]], [[0.4649412005017543, 0.03599127651611266, 0.2849421342796853], [-0.7903070223846702, 1.3343582938702274, 0.33078820660006625], [-0.004089997857936735, 0.37822772435906477, -0.8227179138399797], [-0.11454635400374913, -1.4914438634192537, -1.0933658157048933], [0.3066609749994623, 0.12954327783472816, 0.7543774163599632], [-0.6539197113735622, -0.10211207577058908, -0.6794628696491911]]], "dtype": "float64"}, {"__nd__": [[0.26395801331764274, 0.7900763964031752, 1.075993048013171, 0.5764504671662501], [-0.7898589394360896, 0.21316320919385914, -1.0364885010666045, -1.9214488639194578], [0.17315780312533796, 0.08208643329512258, 0.7411002135780272, 0.8213272660461594], [-1.2388105107914116, -0.9146661842117264, -0.01304258968200663, 0.0636571576750391], [0.39916838642744834, -1.3178111882476198, -0.26341896603411485, 0.6289007047356723], [1.8828318305090208, -1.2688267061761878, 1.605264421945028, -0.3019381696531152], [-1.4603236761927478, 1.3134790017289018, -1.0860508363919765, -1.187252811442023], [0.6983133467975097, 1.0149434288257406, 0.04542681511723151, -0.7440102448727568]], "dtype": "float64"}, {"__nd__": [[-1.5148690288413447, 0.08267143613979593, -2.3637042226219442, -1.482310835296758], [1.0308399930068024, 0.5897674349683752, 1.189474706490783, 0.37670143252552796], [-1.1753494116182133, 0.43693093305720426, -1.8453704739528716, 0.6581871650214526], [1.3261683934876929, -1.0405556845107033, 0.986209741497253, -0.0029098795250363098], [-0.20389078034935193, -2.0172003657999125, -0.20042432191697002, 0.8230373717303825], [-1.7599184629959224, -0.5283723573590631, -0.7507297210471999, -1.6745860195820719], [0.997359557033254, -0.02731258298736381, -0.004999852401579111, 0.9364438543837886], [-0.7983181945814336, 3.2085921360327316, -0.5053469144762512, -1.2078464166591396]], "dtype": "float64"}, {"__nd__": [0.5913527431653398, 0.34000365905220237, -0.9962452593378184, -0.9367339198202601, 0.8025326480832135, 0.26687076455047115, 0.39078380275494706, 0.8495445985078055], "dtype": "float64"}, {"__nd__": [[true, true, true, true, true, true], [true, true, true, false, false, false], [true, false, false, false, false, false]], "dtype": "bool"}], "e": {"__tup__": [{"__nd__": [[[0.7833083538894348, 0.46279655822860283, 0.12031075118840341], [0.5608993046279579, 0.2384525024797769, 0.3208261711806278], [0.9997433899203402, 0.7162998881281532, -0.1455318461849571], [0.7790134650757776, 0.48037065404868284, 0.058899159316164254]], [[0.2424153629913274, 0.021512677115122152, 0.285084514611379], [-0.2030505067822717, 0.2796148783801031, 0.27932849627368583], [0.2090045681496173, 0.04834093175631503, 0.25393978383723226], [0.2995352126693714, -0.082493134916432, 0.5773691668426125]], [[0.4649412005017543, 0.03599127651611266, 0.2849421342796853], [0.4649412005017543, 0.03599127651611266, 0.2849421342796853], [0.4649412005017543, 0.03599127651611266, 0.2849421342796853], [0.4649412005017543, 0.03599127651611266, 0.2849421342796853]]], "dtype": "float64"}, {"__nd__": [[[0.1713332946233136, 0.34816845235942, 0.23231299827707189, 0.054843376195844146, 0.08091458595556683, 0.1124272925887836], [0.0596076420436959, 0.5454328317300313, 0.13067641617329703, 0.06445871076351216, 0.13547733694783842, 0.06434706234162513], [0.3590387696684395, 0.13090921595769056, 0.22766530698027626, 0.059289666565586804, 0.03061492739192636, 0.19248211343608063], [0.2464137480578141, 0.33497348019323675, 0.12149120514215458, 0.05649847426932094, 0.029078151188444968, 0.21154494114902886]], [[0.25997897589944746, 0.10752000457549366, 0.632501019525059, 0.0, 0.0, 0.0], [0.13809345891179536, 0.5127095235145117, 0.34919701757369287, 0.0, 0.0, 0.0], [0.26911809230508704, 0.13974810567323487, 0.5911338020216782, 0.0, 0.0, 0.0], [0.10207565028883692, 0.038116136200974225, 0.8598082135101888, 0.0, 0.0, 0.0]], [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.621765797295769, 1.246840633654848, 0.01709578817885511], [0.7108683973878703, -0.9244765514585456, 0.7468294777927093], [0.27711759969086525, 2.4762073323741025, -0.15000838401427574], [1.532205838061597, 0.6430677931202534, 1.5768531664420402], [0.24867335439343105, 0.49467080845503375, -2.201864880558068]]], "dtype": "float64"}, {"__nd__": [[[-1.2614710169282073, 1.0599310868487861, 0.43970311822212305], [0.7889237041384558, -0.24403664416520585, 0.8018409941574018]]], "dtype": "float64"}, {"__nd__": [[[-0.3383093041669525, -0.038959626361131545, 1.2324993246268356, 1.0950714464789963], [-0.41731182913013487, 0.06914458727381313, 0.2545087998091163, -0.12289066149392434]]], "dtype": "float64"}, {"__nd__": [[-0.08211074196462763, -0.9231190193770936, 0.5842838306753202], [-1.025925409448804, -0.9446524199043005, 1.2855982541642956]], "dtype": "float64"}, {"__nd__": [[1.841137467508027, 1.8604995331973844, 0.3696952948661359], [-0.5179529250903714, 0.7576761265691543, 1.5643556597096155]], "dtype": "float64"}, {"__nd__": [0.13373735961503957, -0.38578683935887204], "dtype": "float64"}, {"__nd__": [[1, 1]], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[-0.3864352177404151, 0.026894142803171366, 0.6367375120141693, 0.3531263102776682], [-0.3787709032173089, 0.016406570476117044, 0.7316158337679863, 0.47128512453177357], [-0.38124814979195726, 0.019796345588457175, 0.7009494255896104, 0.4330940355579266], [-0.3838831545796741, 0.023401991365388914, 0.6683300920121411, 0.39247082779004855], [-0.38000310593837666, 0.018092672360353657, 0.7163621113531584, 0.4522885643040962]]], "dtype": "float64"}, {"__nd__": [[[0.3908306905900688, 0.6091693094099311], [0.48784422941908834, 0.5121557705809117], [0.45648768004547174, 0.5435123199545282], [0.423134255089183, 0.576865744910817], [0.4722472251253392, 0.5277527748746609]]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(additive_attention, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")